# Visualizing Machinery Representation

{py:mod}`claveles.machinery.visualize` draws a machinery representation on the macronode lattice. It differs
from the {ref}`graph drawing <sec:viz-graph-options>` in what there is to show: a machinery
representation has no operations, only the four measurement angles of each macronode and the macronodes
marked for readout.

* {py:func}`~claveles.machinery.visualize.make_figure` returns a {py:class}`matplotlib.figure.Figure`, which
  Jupyter displays and {py:func}`matplotlib.pyplot.show` shows.
* {py:func}`~claveles.machinery.visualize.savefig` writes one straight to a file.

Both take the same keyword options, listed in {ref}`sec:viz-machinery-options`.

In [ ]:
# Draw the figures on this page as SVG so that they stay sharp at any zoom level.
# The cell itself is hidden from the built page by the `remove-cell` tag.
%config InlineBackend.figure_formats = ["svg"]

## A sample machinery representation

We build it from a graph representation with {py:meth}`~claveles.machinery.MachineryRepr.from_graph_repr`:

In [ ]:
from claveles.graph import GraphRepr
from claveles.graph.constant import BLANK_MODE
from claveles.machinery import MachineryRepr

graph = GraphRepr(5, 5)

In [ ]:
from math import pi

from claveles.feedforward import feedforward
from claveles.graph import Wiring, ops


@feedforward
def add_pi(x: float) -> float:
    from math import pi

    return x + pi


@feedforward
def square(x: float) -> float:
    return x * x


@feedforward
def cube(x: float) -> float:
    return x * x * x


graph.place_operation(ops.Initialization((0, 1), 0.0, (0, BLANK_MODE)))
graph.place_operation(ops.Initialization((1, 0), 0.0, (BLANK_MODE, 1)))
graph.place_operation(ops.Initialization((2, 0), 0.0, (BLANK_MODE, 2)))
graph.place_operation(ops.BeamSplitter((1, 1), 0.0, 0.0, displacement_k_minus_n=(1.0, -1.0), swap=True))
graph.place_operation(ops.TwoModeShear((2, 1), 0.0, 0.0, displacement_k_minus_1=(-1.0, 1.0), swap=True))
graph.place_operation(ops.PhaseRotation((3, 1), pi / 2, swap=True))
graph.place_operation(ops.Measurement((1, 2), 0.0))  # Measure mode 0
m0 = graph.get_mode_measured_value(0)
graph.place_operation(Wiring((2, 2), swap=True))
graph.place_operation(ops.ControlledZ((3, 2), 0.0, swap=False))
graph.place_operation(ops.Measurement((4, 2), 0.0))  # Measure mode 1
m1 = graph.get_mode_measured_value(1)
graph.place_operation(
    ops.ShearPInvariant((3, 3), square(m0), displacement_k_minus_n=(add_pi(m0), cube(m1)), swap=True)
)
graph.place_operation(ops.Measurement((4, 3), 0.0))  # Measure mode 2

In [ ]:
machinery = MachineryRepr.from_graph_repr(graph)

Drawing it:

In [ ]:
from claveles.machinery.visualize import make_figure

make_figure(machinery);

{py:func}`~claveles.machinery.visualize.savefig` writes the same figure to a file:

In [ ]:
from claveles.machinery.visualize import savefig

savefig(machinery, "claveles_example.png")

(sec:viz-machinery-options)=

## Options

### The lattice and the figure


| Option | Default | Description |
| --- | --- | --- |
| `title` | `""` | Title written above the figure. Empty means no title. |
| `scale` | `2.0` | Scale of the figure size. Raise it when the image comes out too small to read. |
| `fontsize` | `7.0` | Font size of every label except the feedforward ones. |
| `macronode_radius` | `1.0` | Radius of the large circle standing for a macronode. |
| `macronode_edge_color` | `lightgray` | Colour of that circle's outline. |
| `micronode_radius` | `0.2` | Radius of the four small circles inside a macronode. |
| `micronode_edge_color` | `lightgray` | Colour of their outlines. |
| `highlight_macronode_edge_color` | `{}` | Macronodes to outline in a colour of their own, as a dictionary from `(h, w)` coordinate to colour. It is laid over the `readout_edge_color` outlines below, so an entry here wins for its macronode. |
| `show_displacement` | `True` | Whether the displacement applied to each teleported mode is written beside it. |
| `displacement_text_format` | a two-row column vector | Format string the two displacements are written with, taking the `x` and `p` components in that order. A component that comes from a measurement is written as the symbol standing for it (`d_x^{-1}`, `d_p^{-N}`, and so on) rather than as a number. |
| `show_feedforward` | `True` | Whether feedforward is drawn at all. |
| `show_feedforward_param_label` | `True` | Whether each feedforward arrow is labelled with the parameter it drives. |
| `feedforward_param_fontsize` | `7.0` | Font size of that label. |
| `feedforward_arrow_style` | `->` | Matplotlib arrow style of the feedforward arrows. |
| `feedforward_line_style` | `-` | Line style of the feedforward connections. |
| `feedforward_arrow_color` | `lightgreen` | Their colour, in any format matplotlib accepts. |

### The macronodes

| Option | Default | Description |
| --- | --- | --- |
| `measurement_color` | `black` | Fill colour of a macronode that measures its modes, that is one whose four angles are equal. |
| `operation_color` | `white` | Fill colour of every other macronode. |
| `readout_edge_color` | `red` | Outline colour of the macronodes in {py:attr}`~claveles.machinery.MachineryRepr.readout_macronode_indices`, whose measured values are reported back. |
| `micronode_color` | `lightgray` | Fill colour of the four micronodes. |
| `lattice_edge_color` | `lightgray` | Colour of the lattice edges, that is of the entanglement between macronodes. |
| `lattice_edge_linestyle` | `:` | Line style of those edges. |
| `measurement_params_format` | `${:.2f}$` | Format string the single shared angle of a measuring macronode is written with. |
| `manual_params_format` | `$({:.2f},{:.2f},{:.2f},{:.2f})$` | Format string the four angles of any other macronode are written with. An angle that comes from a measurement is written as the symbol standing for it rather than as a number. |

The example below sets three of them at once:

In [ ]:
make_figure(
    machinery,
    measurement_color="gray",
    operation_color="blue",
    readout_edge_color="yellow",
);

{py:func}`~claveles.machinery.visualize.savefig` takes all of them too:

In [ ]:
savefig(
    machinery,
    "claveles_viz_machinery_repr.png",
    measurement_color="gray",
    operation_color="blue",
    readout_edge_color="yellow",
)

## Visualizing the compiled machinery

Before execution, the client compiles a graph to the target hardware size with
{py:class}`~claveles.client.OptQCCompiler`. Reproducing that step locally and drawing the result is how you
see what the machine will actually be driven with:

In [ ]:
from claveles.client import OptQCCompiler

compiler = OptQCCompiler()
compiled_machinery = compiler.graph_to_machinery(graph)
make_figure(compiled_machinery);